# Solutions · 08 · The exothermic CSTR: multiple steady states and stability

Worked solutions to the exercises of [notebook 08](../notebooks/08_cstr_multiple_steady_states.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engreact is missing): install it from GitHub
    import engreact
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engreact"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engreact import energy
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

In [2]:
k0, Ea, dH, C_A0, cp_vol, kappa, T_a = 5.0e7, 68e3, -250e3, 2500.0, 3.6e6, 0.8, 295.0

## Exercise 1

Halve the residence time to 120 s and repeat the ignition-extinction map. Does multiplicity survive? Now double the feed concentration instead. Explain both results with the adiabatic temperature rise and the Damkohler number.

In [3]:
for tau, C0 in ((120.0, C_A0), (240.0, C_A0), (240.0, 2 * C_A0)):
    T0s = np.linspace(260, 360, 51)
    n = np.array([len(energy.cstr_steady_states(k0, Ea, dH, C0, T0, tau, cp_vol, kappa, T_a, T_range=(250.0, 900.0))["T"]) for T0 in T0s])
    band = T0s[n == 3]
    print(f"tau = {tau:.0f} s, C_A0 = {C0:.0f} mol/m3 (Delta T_ad = {energy.adiabatic_temperature_rise(dH, C0, cp_vol):.0f} K): " + (f"multiplicity for T0 in [{band.min():.0f}, {band.max():.0f}] K" if band.size else "none"))

tau = 120 s, C_A0 = 2500 mol/m3 (Delta T_ad = 174 K): multiplicity for T0 in [322, 344] K
tau = 240 s, C_A0 = 2500 mol/m3 (Delta T_ad = 174 K): multiplicity for T0 in [300, 328] K
tau = 240 s, C_A0 = 5000 mol/m3 (Delta T_ad = 347 K): multiplicity for T0 in [260, 310] K


Halving the residence time (halving the Damkohler number) moves the multiplicity to higher feed
temperatures but does not remove it; doubling the feed concentration doubles the adiabatic rise and widens
the band. Multiplicity needs a large heat release relative to the cooling and a rate that can outrun it; both
levers appear in the dimensionless groups $\Delta T_{ad} E_a/RT^2$ and $k\tau$.

## Exercise 2

Find the feed temperature range over which the reactor can be operated on the *ignited* branch at 240 s with kappa = 0.8, and the start-up procedure (feed temperature schedule) that gets it there from a cold start without exceeding 420 K.

In [4]:
T0s = np.linspace(270, 360, 91)
ignited = [(T0, ss["T"][-1], ss["X"][-1]) for T0 in T0s for ss in [energy.cstr_steady_states(k0, Ea, dH, C_A0, T0, 240.0, cp_vol, kappa, T_a, T_range=(250.0, 900.0))] if ss["X"][-1] > 0.5 and ss["stable"][-1]]
lo, hi = ignited[0][0], ignited[-1][0]
print(f"the ignited branch exists and is stable for T0 from {lo:.0f} K upward (below, the reactor extinguishes); at T0 = 300 K it sits at {[t for t in ignited if abs(t[0]-300) < 0.6][0][1]:.0f} K")
T_max_allowed = 420.0
ok = [t for t in ignited if t[1] <= T_max_allowed]
print(f"staying below {T_max_allowed:.0f} K on the ignited branch requires T0 <= {ok[-1][0]:.0f} K -> operate with T0 between {lo:.0f} and {ok[-1][0]:.0f} K")
print("start-up: feed at >= 330 K (above the ignition point) until the reactor is hot, then lower the feed to the operating value: the hot state persists down to the extinction limit")

the ignited branch exists and is stable for T0 from 300 K upward (below, the reactor extinguishes); at T0 = 300 K it sits at 376 K
staying below 420 K on the ignited branch requires T0 <= 350 K -> operate with T0 between 300 and 350 K
start-up: feed at >= 330 K (above the ignition point) until the reactor is hot, then lower the feed to the operating value: the hot state persists down to the extinction limit


The ignited branch is stable from about 300 K of feed temperature upward, and stays below 420 K only up to a
certain feed temperature: the operating window. Because the ignited state persists once reached, the start-up
recipe is "heat to ignite, then cool to operate" - the hysteresis working for the operator instead of
against them.

## Exercise 3

**Implement it yourself:** write the slope stability test dG/dT < dR/dT with a finite difference and apply it to the three steady states at T0 = 315 K; compare with the library's verdicts.

In [5]:
ss = energy.cstr_steady_states(k0, Ea, dH, C_A0, 315.0, 240.0, cp_vol, kappa, T_a, T_range=(250.0, 900.0))
dR = cp_vol * (1 + kappa)
for T_ in ss["T"]:
    G = lambda T: float(energy.cstr_heat_curves(k0, Ea, dH, C_A0, 315.0, 240.0, cp_vol, T, kappa, T_a)["G"])
    dG = (G(T_ + 0.01) - G(T_ - 0.01)) / 0.02
    print(f"T = {T_:.0f} K: dG/dT = {dG/1e3:.1f} kJ/(m3 K), dR/dT = {dR/1e3:.1f} kJ/(m3 K) -> {'stable' if dG < dR else 'UNSTABLE'}")
print("library:", ss["stable"])

T = 310 K: dG/dT = 2046.3 kJ/(m3 K), dR/dT = 6480.0 kJ/(m3 K) -> stable
T = 349 K: dG/dT = 10361.4 kJ/(m3 K), dR/dT = 6480.0 kJ/(m3 K) -> UNSTABLE
T = 395 K: dG/dT = 2244.8 kJ/(m3 K), dR/dT = 6480.0 kJ/(m3 K) -> stable
library: [True, False, True]


The slope test is a finite difference of the generation curve against the constant slope of the removal
line; the middle state fails it. It is a necessary condition (the full test involves the concentration
dynamics too), but for this first-order system it is also sufficient.